# Train the L3C baseline on a GPU (Colab)

Trains the 16-bit greyscale L3C baseline (`baselines/l3c/l3c.py`) for 150,000 steps on the same
training patients as the paper, then runs the bit-exact evaluation on the 258 held-out test images.

**How to run:** Runtime → Change runtime type → **GPU** (T4 is fine, A100/L4 is faster) → Runtime → **Run all**.
Allow Google Drive access when asked.

Checkpoints, data cache and results are kept in Google Drive folder **`MyDrive/medical-wavelet-codec-L3C/`**:

| file | what |
|---|---|
| `data/` | cached IDC images, so a restart does not download them again |
| `l3c_train_state.pt` | full training state, saved every 1,000 steps |
| `l3c.pt` | the trained model (updated every 1,000 steps, final at step 150,000) |
| `train.log` | training log |
| `results/` | bit-exact evaluation of L3C on the 258 test images |

**If Colab disconnects:** reconnect and press Run all again. Training continues from the last saved step
(the resumed run gives exactly the same weights as an uninterrupted one), and the evaluation continues
from the last finished image.

In [ ]:
import os, subprocess, sys, json, hashlib, shutil, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
D = Path('/content/drive/MyDrive/medical-wavelet-codec-L3C')
D.mkdir(parents=True, exist_ok=True)
STEPS, BATCH, LR = 150_000, 16, 1e-4
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to GPU')

## 1. Code and packages

In [ ]:
CODE = Path('/content/code')
shutil.rmtree(CODE, ignore_errors=True)
!git clone -q https://github.com/AI-Maddy/medical-wavelet-codec /content/code
!pip -q install "constriction>=0.4" "imagecodecs>=2024.1.1" "pydicom>=3.0" "idc-index>=0.8" "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.0" "pylibjpeg-openjpeg>=2.0"
%cd /content/code

## 2. Data (same patients as the paper)

Downloaded once from the NCI Imaging Data Commons (about 20 minutes) and cached in Drive.
Every file is checked against the hashes of the data the paper used.

In [ ]:
DATA = Path('/content/data')
manifest = json.loads((CODE / 'notebooks' / 'data_manifest.json').read_text())
import numpy as np
def check(root):
    bad = []
    for rel, exp in manifest.items():
        f = root / rel
        if not f.exists():
            bad.append(rel + ' missing'); continue
        a = np.load(f)['images']
        if hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest() != exp['sha256']:
            bad.append(rel + ' differs')
    return bad
if not (D / 'data' / 'DONE').exists():
    !python scripts/fetch_idc.py --out /content/data 2>&1 | tail -5
    bad = check(DATA)
    assert not bad, f'downloaded data differ from the paper data: {bad[:5]} ... ({len(bad)} files)'
    shutil.copytree(DATA, D / 'data', dirs_exist_ok=True)
    (D / 'data' / 'DONE').write_text('ok')
else:
    shutil.copytree(D / 'data', DATA, dirs_exist_ok=True)
    bad = check(DATA)
    assert not bad, bad[:5]
n_test = sum(v['n'] for k, v in manifest.items() if '/test_' in k)
print('data OK:', len(manifest), 'files,', n_test, 'test images')

## 3. Train (150,000 steps)

Prints one line every 50 steps (`bpp` is the training loss in bits per pixel). Time per step depends on the
GPU; the first lines show the speed. Colab may stop a long session: then just Run all again.

In [ ]:
!python baselines/l3c/l3c.py train --data /content/data --steps {STEPS} --batch {BATCH} --lr {LR} --crop 128 --threads 2 --save-every 1000 --out "{D}/l3c.pt" --ckpt "{D}/l3c_train_state.pt" 2>&1 | grep -v Warning | tee -a "{D}/train.log"
import torch
ck = torch.load(D / 'l3c.pt', map_location='cpu', weights_only=False)
assert ck['step'] == STEPS, f"training stopped at step {ck['step']}: run all again to continue"
print('trained model saved:', D / 'l3c.pt', 'step', ck['step'])

## 4. Bit-exact evaluation on the 258 test images

Every image is encoded with L3C, decoded again and compared pixel for pixel; a mismatch stops the run.
This runs on the CPU (as in the paper) and takes a few hours; it continues where it stopped after a disconnect.

In [ ]:
R = D / 'results'
R.mkdir(exist_ok=True)
!mkdir -p /content/m && cp "{D}/l3c.pt" /content/m/l3c.pt
!python scripts/evaluate.py --data /content/data --models /content/m --out "{R}" --configs l3c --threads 2 2>&1 | grep -v Warning | tail -3
!python scripts/summarize.py --results "{R}" | grep -i -B3 l3c
import pandas as pd
d = pd.read_csv(R / 'per_image.csv')
print(len(d), 'images evaluated,', int(d.lossless.sum()), 'decoded bit-exactly')